# Feature Maps, Kernels and the QML Bridge
Predict first, run the calculations and explain the differences. Data are generated locally with fixed seeds; reported scores describe these teaching examples, not general performance guarantees.


In [ ]:
%pip -q install "numpy>=2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4"


In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score


## Check an explicit polynomial feature map
A PSD matrix need not have positive entries, although this squared-dot-product example does.


In [ ]:
def phi(X):
    X=np.asarray(X)
    return np.column_stack([X[:,0]**2,np.sqrt(2)*X[:,0]*X[:,1],X[:,1]**2])
X=np.array([[1,2],[2,1],[-1,1]],float)
K=(X@X.T)**2
assert np.allclose(K,phi(X)@phi(X).T)
assert np.linalg.eigvalsh(K).min()>-1e-10
assert np.isclose(K[0,1],16)


## An analytic product-state fidelity kernel
Each scaled coordinate prepares Ry(xj)|0>. Product-state fidelities multiply. This analytic classical calculation is a bridge example, not a quantum hardware experiment or advantage claim.


In [ ]:
def state_kernel(X,Y):
    return np.prod(np.cos((X[:,None,:]-Y[None,:,:])/2)**2,axis=2)
assert np.isclose(state_kernel(np.array([[0.]]),np.array([[np.pi]]))[0,0],0)
Kq=state_kernel(X,X)
assert np.allclose(np.diag(Kq),1)
assert np.linalg.eigvalsh(Kq).min()>-1e-10


## Compare procedures on the same data
All methods see the same feature coordinates and split. Scaling is fitted only within training folds. Search budgets differ and are stated: linear and analytic state kernels use three C values, RBF uses nine C/gamma pairs. This is a teaching comparison, not a definitive benchmark.


In [ ]:
X,y=make_moons(n_samples=240,noise=.2,random_state=2026)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=42)
cv=StratifiedKFold(4,shuffle=True,random_state=42)
models=[("linear",SVC(kernel="linear"),{"svc__C":[.1,1,10]}),("RBF",SVC(kernel="rbf"),{"svc__C":[.1,1,10],"svc__gamma":[.1,1,10]}),("analytic state kernel",SVC(kernel=state_kernel),{"svc__C":[.1,1,10]})]
baseline=DummyClassifier(strategy="most_frequent").fit(Xtr,ytr)
print("Dummy test accuracy:",baseline.score(Xte,yte))
for name,model,grid in models:
    search=GridSearchCV(make_pipeline(StandardScaler(),model),grid,cv=cv,scoring="balanced_accuracy")
    search.fit(Xtr,ytr)
    pred=search.predict(Xte)
    print(name,"CV choice",search.best_params_,"test accuracy",accuracy_score(yte,pred),"balanced accuracy",balanced_accuracy_score(yte,pred))
    assert np.allclose(search.best_estimator_[0].mean_,Xtr.mean(axis=0))


## Exercises
1. Change the noise level while preserving the comparison protocol.
2. Inspect periodic similarities of the analytic state kernel.
3. Explain which costs an actual circuit-based overlap estimate would introduce.
4. Do not select another kernel based on the displayed test scores; use fresh validation or an independently held evaluation dataset for a new comparison.
References: https://scikit-learn.org/stable/modules/svm.html and https://scikit-learn.org/stable/common_pitfalls.html
